In [0]:
%sql
DROP TABLE IF EXISTS group3_gp.gold.tabel_count;
CREATE OR REPLACE TABLE group3_gp.gold.tabel_count (
    source STRING,
    row_count INT,
    reason STRING
)

In [0]:
%sql
DROP TABLE IF EXISTS group3_gp.gold.dim_zone;
CREATE OR REPLACE TABLE group3_gp.gold.dim_zone AS
SELECT
    l.LocationID AS zone_id,
    COALESCE(l.Zone, z.zone) AS zone_name,
    COALESCE(l.Borough, z.borough) AS borough,
    l.service_zone,
    z.geometry_wkt
FROM group3_gp.bronze.taxi_zones_lookup l
LEFT JOIN group3_gp.bronze.taxi_zones z
    ON l.LocationID = z.LocationID;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

start_date = "2009-01-01"

dim_date = (
    spark.range(1)
    .select(sequence(to_date(lit(start_date)), current_date()).alias("d"))
    .select(explode("d").alias("date"))
    .select(
        col("date"),
        year("date").alias("year"),
        month("date").alias("month"),
        date_format("date", "MMMM").alias("month_name"),
        dayofmonth("date").alias("day"),
        date_format("date", "EEEE").alias("day_of_week_name"),
        dayofweek("date").alias("day_of_week_num"),
        when(dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
        quarter("date").alias("quarter"),
        weekofyear("date").alias("week_of_year")
    )
    .orderBy("date")
)

In [0]:

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.dim_date")